# رینگ رزوناتور Add-Drop (طراحی معمولی) با Meep

همان فناوری نوت‌بوک MZI: Si 220 nm روی SiO2، مدل 2D با ضریب شکست مؤثر، عرض موجبر `0.45 µm`، مود TE.

**آنچه این نوت‌بوک انجام می‌دهد**
1. هندسهٔ رینگ + دو موجبر باس (ورودی/Through پایین، Drop بالا) را می‌سازد.
2. FDTD با منبع مودی و مانیتورهای مودی (Through، Drop، بازتاب) اجرا می‌کند.
3. از طیف: FSR، پهنای باند (FWHM)، ضریب کیفیت Q، نسبت خاموشی و افت درج Drop را استخراج می‌کند.
4. طیف را با مدل کوپلینگ add-drop فیت می‌کند و `κ²` و افت چرخه را می‌دهد.
5. (اختیاری) جاروب gap، نقشهٔ میدان در تشدید، و خروجی GDS.

> در 2D تشدیدها باریک‌اند، پس باند را فقط ۴۰ nm و با تفکیک ~۰٫۱ nm می‌گیریم. ابتدا `QUICK_TEST = True` را امتحان کنید.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
!mamba install -y -q -c conda-forge pymeep scipy matplotlib
!pip install -q gdstk

## ۰) پارامترها

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import meep as mp
from scipy.signal import find_peaks, peak_widths
from scipy.optimize import least_squares

try:
    mp.verbosity(0)
except Exception:
    pass

# ======================= switches =======================
QUICK_TEST = False        # True: coarse + short run, only to check the pipeline
RUN_GAP_SWEEP = False     # True: also simulate several coupling gaps (slow)
PLOT_FIELD = False        # True: extra run at one resonance to plot the field (slow)

RES = 16 if QUICK_TEST else 32          # px/um
NF = 101 if QUICK_TEST else 401         # number of wavelength points
PARITY = mp.EVEN_Z                      # TE-like (same as the MZI notebook)

# ======================= technology (same as MZI notebook) =======================
t_si, n_clad, n_si = 0.22, 1.444, 3.476


def slab_neff(t, n1=n_si, n2=n_clad, lam0=1.55):
    k0 = 2 * np.pi / lam0
    a = k0 * t / 2
    V = a * np.sqrt(n1 ** 2 - n2 ** 2)
    lo, hi = 1e-9, min(V, np.pi / 2 - 1e-9)
    f = lambda u: u * np.tan(u) - np.sqrt(max(V ** 2 - u ** 2, 0.0))
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if f(mid) > 0:
            hi = mid
        else:
            lo = mid
    u = 0.5 * (lo + hi)
    return float(np.sqrt(n1 ** 2 - (u / a) ** 2))


n_core = slab_neff(t_si)
print(f"2D effective core index: {n_core:.4f}")
Si2D = mp.Medium(index=n_core)
SiO2 = mp.Medium(index=n_clad)

# ======================= ring geometry =======================
w = 0.45              # waveguide width (ring and buses)
R_RING = 5.0          # ring centre-line radius (um)
GAP = 0.10            # bus-to-ring gap (um)
L_RING = 2 * np.pi * R_RING

# ======================= spectral window =======================
WL_MIN, WL_MAX = 1.53, 1.57
fmin, fmax = 1 / WL_MAX, 1 / WL_MIN
fmid, fspan = 0.5 * (fmin + fmax), fmax - fmin
fcen_src, fwidth_src = fmid, 0.03
dpml = 1.0
src_size = 1.6

# run control: stop when the monitors' DFT has decayed
DECAY_TOL = 1e-5 if QUICK_TEST else 1e-8     # a loose tolerance stops the run before the ring has rung down
T_MIN = 200 if QUICK_TEST else 1500
T_MAX = 600 if QUICK_TEST else 8000

## ۱) هندسه و شبیه‌ساز

- حلقه: دو `Cylinder` (بیرونی Si، درونی SiO2) به شعاع مرکزی `R`.
- باس پایین: ورودی `P0` (چپ)، خروجی **Through** (راست). باس بالا: خروجی **Drop** (چپ)، چون نور در حلقه پادساعتگرد می‌چرخد.
- نرمال‌سازی با اجرای مرجع (فقط باس پایین، بدون حلقه).

In [ ]:
def build(gap, res, with_ring=True, fcen=fcen_src, fw=fwidth_src):
    yb = R_RING + w + gap                     # bus centre-line |y|
    Lx = R_RING + 2.5
    sx = 2 * (Lx + dpml)
    sy = 2 * (yb + w / 2 + 0.8 + 0.6 + dpml)
    geo = [mp.Block(size=mp.Vector3(mp.inf, w, mp.inf), center=mp.Vector3(0, -yb, 0), material=Si2D)]
    if with_ring:
        geo += [mp.Block(size=mp.Vector3(mp.inf, w, mp.inf), center=mp.Vector3(0, +yb, 0), material=Si2D),
                mp.Cylinder(radius=R_RING + w / 2, height=mp.inf, material=Si2D),
                mp.Cylinder(radius=R_RING - w / 2, height=mp.inf, material=SiO2)]
    XS, XM = -(R_RING + 1.6), R_RING + 1.0
    src = mp.EigenModeSource(mp.GaussianSource(fcen, fwidth=fw), center=mp.Vector3(XS, -yb, 0),
                             size=mp.Vector3(0, src_size, 0), eig_band=1, eig_parity=PARITY, eig_match_freq=True)
    sim = mp.Simulation(cell_size=mp.Vector3(sx, sy, 0), boundary_layers=[mp.PML(dpml)], geometry=geo,
                        sources=[src], default_material=SiO2, resolution=res, dimensions=2)
    pos = {"thru": (+XM, -yb), "drop": (-XM, +yb), "refl": (-XM, -yb)}
    return sim, pos, (sx, sy)


def run_ring(gap, res, nf, with_ring=True):
    t0 = time.time()
    sim, pos, _ = build(gap, res, with_ring)
    keys = ["thru", "drop", "refl"] if with_ring else ["thru"]
    mons = {k: sim.add_mode_monitor(fmid, fspan, nf, mp.ModeRegion(center=mp.Vector3(*pos[k], 0),
                                                                   size=mp.Vector3(0, src_size, 0))) for k in keys}
    sim.run(until_after_sources=mp.stop_when_dft_decayed(tol=DECAY_TOL, minimum_run_time=T_MIN,
                                                         maximum_run_time=T_MAX))
    al = {k: np.array(sim.get_eigenmode_coefficients(m, [1], eig_parity=PARITY).alpha[0]) for k, m in mons.items()}
    fr = np.array(mp.get_flux_freqs(mons["thru"]))
    print(f"   done in {time.time() - t0:.0f} s (sim time {sim.meep_time():.0f})")
    return al, fr


def spectra(gap, res, nf, a_ref):
    al, fr = run_ring(gap, res, nf, True)
    o = np.argsort(1 / fr)                     # ascending wavelength
    lam_ = (1 / fr)[o]
    Tt = (np.abs(al["thru"][:, 0]) ** 2 / np.abs(a_ref) ** 2)[o]
    Td = (np.abs(al["drop"][:, 1]) ** 2 / np.abs(a_ref) ** 2)[o]
    Rr = (np.abs(al["refl"][:, 1]) ** 2 / np.abs(a_ref) ** 2)[o]
    return lam_, Tt, Td, Rr


# ---- geometry preview
fig, ax = plt.subplots(figsize=(6.5, 6.5))
yb_ = R_RING + w + GAP
th = np.linspace(0, 2 * np.pi, 400)
ax.fill(np.r_[(R_RING + w / 2) * np.cos(th), (R_RING - w / 2) * np.cos(th[::-1])],
        np.r_[(R_RING + w / 2) * np.sin(th), (R_RING - w / 2) * np.sin(th[::-1])], color="C0", alpha=0.6)
for s in (-1, 1):
    ax.add_patch(plt.Rectangle((-R_RING - 2.5, s * yb_ - w / 2), 2 * (R_RING + 2.5), w, color="C0", alpha=0.6))
ax.set_aspect("equal"); ax.set_xlabel("x (um)"); ax.set_ylabel("y (um)")
ax.set_title(f"Add-drop ring: R = {R_RING} um, gap = {GAP} um"); plt.show()

## ۲) اجرای مرجع و شبیه‌سازی رینگ

In [ ]:
print("Reference run (straight bus) ...")
al_ref, _ = run_ring(GAP, RES, NF, with_ring=False)
a_ref = al_ref["thru"][:, 0]

print("Ring run ...")
lam, Tt, Td, Rr = spectra(GAP, RES, NF, a_ref)
dB = lambda p: 10 * np.log10(np.maximum(p, 1e-12))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(lam, dB(Tt), label="Through"); ax[0].plot(lam, dB(Td), label="Drop"); ax[0].plot(lam, dB(Rr), label="Reflection")
ax[0].set_xlabel("Wavelength (um)"); ax[0].set_ylabel("Power (dB)"); ax[0].set_ylim(-45, 2)
ax[0].grid(alpha=0.3); ax[0].legend(); ax[0].set_title("Add-drop ring: FDTD")
ax[1].plot(lam, Tt, label="Through"); ax[1].plot(lam, Td, label="Drop"); ax[1].plot(lam, Tt + Td, "k:", label="Through + Drop")
ax[1].set_xlabel("Wavelength (um)"); ax[1].set_ylabel("|S|^2"); ax[1].set_ylim(0, 1.05)
ax[1].grid(alpha=0.3); ax[1].legend(); plt.tight_layout(); plt.show()
print(f"convergence check: max(T+D+R) = {(Tt + Td + Rr).max():.3f} | through max = {dB(Tt.max()):.2f} dB")
if (Tt + Td + Rr).max() > 1.02 or Tt.max() > 1.02:
    print("WARNING: passive device with power > 1 -> the run stopped too early. Tighten DECAY_TOL / raise T_MIN and re-run.")

## ۳) استخراج FSR، Q، نسبت خاموشی

In [ ]:
def neff_curve(freq_list, width, n_core_, res):
    sim = mp.Simulation(cell_size=mp.Vector3(1, 5, 0),
                        geometry=[mp.Block(size=mp.Vector3(mp.inf, width, mp.inf), center=mp.Vector3(),
                                           material=mp.Medium(index=n_core_))],
                        default_material=SiO2, resolution=res, dimensions=2)
    sim.init_sim()
    vol_ = mp.Volume(center=mp.Vector3(), size=mp.Vector3(0, 4, 0))
    return np.array([sim.get_eigenmode(f, mp.X, vol_, 1, mp.Vector3(2.3 * f, 0, 0), parity=PARITY).k.x / f
                     for f in freq_list])

f_sub = np.linspace(fmin, fmax, 7)
neff_sub = neff_curve(f_sub, w, n_core, RES)
lam_sub = 1 / f_sub
ng_all = neff_sub - lam_sub * np.gradient(neff_sub, lam_sub)
neff_c, ng_c = float(np.interp(fmid, f_sub, neff_sub)), float(np.interp(fmid, f_sub, ng_all))
lam_c = 1 / fmid
print(f"n_eff = {neff_c:.4f},  n_g = {ng_c:.3f}  ->  expected FSR = {lam_c**2 / (ng_c * L_RING) * 1e3:.2f} nm "
      f"(ring circumference {L_RING:.2f} um; bend dispersion ignored)")

# ---- resonances from the drop port
pk, _ = find_peaks(Td, prominence=0.1 * Td.max())
print(f"Found {len(pk)} drop resonances at lambda (um):", np.round(lam[pk], 4))
widths = peak_widths(Td, pk, rel_height=0.5)[0]
dlam = np.mean(np.abs(np.diff(lam)))
rows = []
for i, p in enumerate(pk):
    fwhm = widths[i] * dlam
    rows.append((lam[p], fwhm * 1e3, lam[p] / fwhm, Td[p]))
    print(f"  lambda_res = {lam[p]:.4f} um | FWHM = {fwhm*1e3:.3f} nm | Q_loaded = {lam[p]/fwhm:,.0f} | "
          f"drop peak = {dB(Td[p]):.2f} dB")
if len(pk) >= 2:
    print(f"Measured FSR = {np.mean(np.abs(np.diff(lam[pk]))) * 1e3:.2f} nm")
print(f"Through extinction (max/min in window): {dB(Tt.max()) - dB(Tt.min()):.1f} dB")
if len(pk) < 2:
    print("Warning: <2 resonances; widen WL_MIN/WL_MAX (and raise NF) for a direct FSR measurement.")

## ۴) فیت مدل add-drop

برای رینگ متقارن با ضریب عبور خودی `r` و دامنهٔ عبور یک چرخهٔ کامل `a`:

با کوپلرهای یکسان (دامنهٔ عبور `r`، **توان** تزویج `κ²`، که ممکن است `r² + κ² < 1` باشد) و دامنهٔ عبور چرخه `a`، با `ℓ = r² + κ²`:

`T_through = r²(1 − 2ℓa cosφ + ℓ²a²) / D` ، `T_drop = κ⁴ a / D` ، `D = 1 − 2r²a cosφ + r⁴a²`

(برای کوپلر بدون افت `ℓ = 1` و `κ² = 1 − r²` است. این فرمول با بازگشت میدان-دامنه راستی‌آزمایی شده؛ نسخهٔ قبلی با `√a` در صورت Drop اشتباه بود.)

In [ ]:
def ring_model(f, r, k2, a, FSR_f, f_res):
    # two identical couplers: through amplitude r, cross POWER k2 (r^2 + k2 <= 1), round-trip amplitude a
    phi = 2 * np.pi * (f - f_res) / FSR_f
    l_ = r * r + k2
    D = 1 - 2 * r * r * a * np.cos(phi) + (r * r * a) ** 2
    return r * r * (1 - 2 * l_ * a * np.cos(phi) + (l_ * a) ** 2) / D, k2 * k2 * a / D


f_arr = 1 / lam
p0 = pk[np.argmax(Td[pk])]
FSR_f0 = 1 / (ng_c * L_RING)
lb = [0.5, 0.005, 0.3, 0.7 * FSR_f0, f_arr[p0] - 0.3 * FSR_f0]
ub = [0.9999, 0.6, 1.0, 1.3 * FSR_f0, f_arr[p0] + 0.3 * FSR_f0]


def resid(p):
    mt, md_ = ring_model(f_arr, *p)
    return np.r_[mt - Tt, md_ - Td]

fit = least_squares(resid, [0.9, 0.1, 0.95, FSR_f0, f_arr[p0]], bounds=(lb, ub))
r_f, k2_f, a_f, FSR_f_f, f_res_f = fit.x
FSR_nm = (1 / f_res_f) ** 2 * FSR_f_f * 1e3
loss_rt_dB = -20 * np.log10(a_f)
print(f"Fit:  r^2 = {r_f**2:.4f}   kappa^2 = {k2_f:.4f}   coupler loss = {-10*np.log10(r_f**2 + k2_f):.3f} dB per coupler")
print(f"      a = {a_f:.4f}  -> round-trip loss (ring only, couplers excluded) = {loss_rt_dB:.3f} dB  ->  {loss_rt_dB / (L_RING * 1e-4):.1f} dB/cm")
print(f"      FSR = {FSR_nm:.2f} nm   |   ng from fitted FSR = {(1 / f_res_f) ** 2 / (FSR_nm * 1e-3 * L_RING):.3f}  (mode solver: {ng_c:.3f})")

mt, mdp = ring_model(f_arr, *fit.x)
plt.figure(figsize=(8, 4))
plt.plot(lam, Tt, "C0-", label="FDTD through"); plt.plot(lam, Td, "C1-", label="FDTD drop")
plt.plot(lam, mt, "C0--", label="model through"); plt.plot(lam, mdp, "C1--", label="model drop")
plt.xlabel("Wavelength (um)"); plt.ylabel("|S|^2"); plt.grid(alpha=0.3); plt.legend(ncol=2)
plt.title("FDTD vs fitted add-drop model"); plt.show()

## ۵) اختیاری: جاروب gap، میدان در تشدید، خروجی‌ها

In [ ]:
if RUN_GAP_SWEEP:
    GAPS = [0.08, 0.10, 0.15] if not QUICK_TEST else [0.10, 0.15]
    plt.figure(figsize=(8, 4))
    for g in GAPS:
        al_g, _ = run_ring(g, RES, NF, with_ring=False)      # reference depends on the cell, so re-run it
        lam_g, Tt_g, Td_g, _ = spectra(g, RES, NF, al_g["thru"][:, 0])
        p_g, _ = find_peaks(Td_g, prominence=0.05 * max(Td_g.max(), 1e-3))
        if len(p_g):
            q = lam_g[p_g[0]] / (peak_widths(Td_g, p_g, rel_height=0.5)[0][0] * dlam)
            print(f"gap {g*1e3:.0f} nm: Q_loaded ~ {q:,.0f}, drop peak {dB(Td_g[p_g[0]]):.2f} dB")
        plt.plot(lam_g, dB(Td_g), label=f"drop, gap {g*1e3:.0f} nm")
    plt.xlabel("Wavelength (um)"); plt.ylabel("Power (dB)"); plt.ylim(-40, 2); plt.grid(alpha=0.3); plt.legend()
    plt.title("Coupling gap sweep"); plt.show()

if PLOT_FIELD:
    f0 = f_res_f
    sim_f, pos_f, (sx_f, sy_f) = build(GAP, RES, True, fcen=f0, fw=0.004)
    dft = sim_f.add_dft_fields([mp.Ey], f0, 0, 1, where=mp.Volume(center=mp.Vector3(),
                               size=mp.Vector3(sx_f - 2 * dpml, sy_f - 2 * dpml, 0)))
    sim_f.run(until_after_sources=600)
    fld = np.real(sim_f.get_dft_array(dft, mp.Ey, 0))
    vm = np.max(np.abs(fld))
    plt.figure(figsize=(7, 7))
    plt.imshow(fld.T, origin="lower", cmap="RdBu", vmin=-vm, vmax=vm,
               extent=[-(sx_f - 2 * dpml) / 2, (sx_f - 2 * dpml) / 2, -(sy_f - 2 * dpml) / 2, (sy_f - 2 * dpml) / 2])
    plt.title(f"Re(Ey) at resonance {1 / f0:.4f} um"); plt.xlabel("x (um)"); plt.ylabel("y (um)"); plt.show()

np.savez("ring_resonator_results.npz", wavelength_um=lam, through=Tt, drop=Td, reflection=Rr,
         fit_r=r_f, fit_a=a_f, fit_FSR_nm=FSR_nm, R_um=R_RING, gap_um=GAP, width_um=w)

try:
    import gdstk
    Lb_ = R_RING + 8
    yb_ = R_RING + w + GAP
    shapes = [gdstk.ellipse((0, 0), R_RING + w / 2, inner_radius=R_RING - w / 2, tolerance=1e-3, layer=2, datatype=0),
              gdstk.rectangle((-Lb_, -yb_ - w / 2), (Lb_, -yb_ + w / 2), layer=2, datatype=0),
              gdstk.rectangle((-Lb_, yb_ - w / 2), (Lb_, yb_ + w / 2), layer=2, datatype=0)]
    lib = gdstk.Library(); cg = lib.new_cell("RING_ADD_DROP")
    for p_ in gdstk.boolean(shapes, [], "or", precision=1e-3, layer=2, datatype=0):
        cg.add(p_)
    lib.write_gds("ring_add_drop.gds")
    print("Saved ring_resonator_results.npz and ring_add_drop.gds")
except Exception as e:
    print("GDS export skipped:", e)